## AureaMind - Documents\AureaMind\data\raw

In [1]:
"""
AureaMind - Synthetic Data Generator
-----------------------------------
This script generates all synthetic datasets for the AureaMind project.
Running this file will recreate the complete data universe of the platform.

Author: Cindy Yohana Gutierrez Nestor
Project: AureaMind – Wellbeing Intelligence Platform for Women
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

In [3]:
#                   Global configuration:
# ---------------------------------------------------------------

# Cambiar estos datos por la cantidad de observaciones que necesites para cada tabla
N_USERS = 1000
print(f'{N_USERS} observations are being simulated')

# Cambiar estos datos por las fechas que necesites
START_DATE = datetime(2025,7,1)
FINISH_DATE = datetime(2025,12,31)

print(f'Start Date: {START_DATE} \nFinish Date: {FINISH_DATE}')

# Definiendo un path en donde se guardarán los datasets en formato CSV para que sea reutilizable:
OUTPUT_PATH = 'C:/Users/vorte/Documents/AureaMind/data/raw'
print(f'Los Datasets creados se guardaran en {OUTPUT_PATH}')

1000 observations are being simulated
Start Date: 2025-07-01 00:00:00 
Finish Date: 2025-12-31 00:00:00
Los Datasets creados se guardaran en C:/Users/vorte/Documents/AureaMind/data/raw


### Generating functions:

In [5]:
# Creating USERS dataframe:
def generate_users(n_users = N_USERS):
    # Creando opciones para las variables que lo necesiten:
    countries = ["Mexico", "USA", "Colombia", "Argentina", "Chile", "Canada","Brazil"]
    education_levels = ["High School", "Bachelor", "Master", "PhD"]
    employment_statuses = ["Employed", "Unemployed", "Student", "Freelancer"]
    relationship_statuses = ["Single", "In Relationship", "Married"]
    program_types = ["Basic", "Therapy", "Coaching", "Holistic"]
    singup_date = [START_DATE + timedelta(days = int(x)) for x in np.random.uniform(0,180,n_users)]
    dropout = np.random.choice([True, False], size=n_users, p=[0.3, 0.7])

    #creating dropout dates in case of dropouts = True
    dropout_dates = []
    for i in range(n_users):
        signup = singup_date[i]
        is_dropout = dropout[i]
        if is_dropout == True:
            dropout_date = singup_date[i] + timedelta(days=int(np.random.uniform(20, 180)))
            if dropout_date > FINISH_DATE:
                dropout_date = FINISH_DATE
        else:
            dropout_date = FINISH_DATE
        dropout_dates.append(dropout_date)
        
    users = pd.DataFrame(
        {'user_id': np.arange(1, n_users +1),
         'age': np.random.randint(18,60, size = n_users),
         "country" : np.random.choice(countries, size = n_users),
         'education_level' : np.random.choice(education_levels, n_users),
         'employment_status' : np.random.choice(employment_statuses, size= n_users),
         'relationship_status': np.random.choice(relationship_statuses, size=n_users),
         'has_children': np.random.choice([True,False], n_users),
         'baseline_stress': np.round(np.random.uniform(2,9,n_users)),
         'baseline_anxiety': np.round(np.random.uniform(2,9,n_users)),
         'baseline_depression': np.round(np.random.uniform(2,8,n_users)),
         'signup_date': singup_date,
         'program_type': np.random.choice(program_types,n_users),
         'dropout': dropout,
         'dropout_date' : dropout_dates
        })
    return users

In [122]:
# Creading second datasframe ASSESSMENTS
def generate_assessments(users_df, max_assessments_per_user = 10):    #para obtener 10 sesiones por persona maximo
    data= []
    assessment_id = 1

    for _, user in users_df.iterrows():           #loop para iterar la fecha de entrada de cada usuaria
        start =user['signup_date']
        n_assessments = np.random.randint(3,max_assessments_per_user+1)

        for i in range(n_assessments):           # loop para escribir la fecha cada sesion de cada usuaria
            date = start + timedelta(days =np.random.randint(7,60))
            
            # añadiendo toda la data a cada fila de cada sesion:
            data.append({
                'assessment_id': assessment_id,
            'user_id': user['user_id'],
            'date': date,
            'stress_score': np.round(np.random.uniform(2,10)),
            'anxiety_score': np.round(np.random.uniform(2,9)),
            'depresionscore':np.round(np.random.uniform(2,10)),
            'sleep_quality':np.round(np.random.uniform(2,10)),
            'mood_score': np.round(np.random.uniform(2,10))
    })
            assessment_id += 1
    #Creando el Dataframe
    return pd.DataFrame(data)
    

In [19]:
# Generating the thirth DataFrame "Sessions"

def generate_sessions(users_df, max_sessions_per_user = 24):
    session_id = 1
    data = [] # Here you store the future data for the dataframe

    for i, user in users_df.iterrows():
        start_date = user['signup_date']
        n_sessions = np.random.randint(6, max_sessions_per_user +1)
        current_date = start_date + timedelta(days=np.random.randint(7, 14))

        for n_session in range(n_sessions):

            # Adding the rest of the fields for each session
            data.append({
                'session_id': session_id,
                'user_id': user['user_id'],
                'date': current_date,
                'session_type': np.random.choice(['Yoga','Psicotherapy','Aqupuncture','Massage']),
                'duration_min': np.random.choice([45,60,90]),
                'attended': np.random.choice([True, False] p =[0.3, 0.7]),
                'therapist_id': np.random.randint(1,9),       # Imagine 8 different therapists
                'topic' : np.random.choice(['Childhood','Work-Relationship','Emotional Situation', 'Close Family Circle', 'Mother', 'Father', 'Other'])
            })
            # updating session_id
            session_id += 1
            # Updating next session to be onthe following weeks
            current_date += timedelta(days=np.random.randint(7, 15))
    
    sessions = pd.DataFrame(data)
    return sessions

In [87]:
# Generating the fourth DataFrame "engagement"

def generate_engagement(users_df, max_app_openings = 4): # 180 days = 6 months
    data = []
    
    for i, user in users_df.iterrows():
        user_id = user['user_id']
        current_date = user['signup_date']
        finish_date = user['dropout_date']
        valid_days = (finish_date - current_date).days + 1

        for day in range(valid_days):        # each day of the user's active period
            date = current_date
            app_openings = int(np.round(np.random.uniform(1, max_app_openings +1)))  # the user opens the app once at day o 4 times at day
        
            messages_sent = 0
            for opening in range(app_openings):
                messages_sent += np.random.randint(1, 4)
        
            exercises_completed = np.random.randint(1,4)
            meditations_completed = np.random.randint(1,3)
            journals_written = np.random.randint(1,3)

            data.append({
                    'user_id':user_id,
                    'date': date,
                    'messages_sent':messages_sent,
                    'exercises_completed':exercises_completed,
                    'meditations_completed':meditations_completed,
                    'journals_written':journals_written
            })
            current_date += timedelta(days=1)  # the user opens the app every day

    engagement = pd.DataFrame(data)
    return engagement


In [136]:
g = generate_engagement(users)
g

,user_id,date,messages_sent,exercises_completed,meditations_completed,journals_written
0,1,2025-12-14,9,1,2,2
1,1,2025-12-15,7,2,2,1
2,1,2025-12-16,8,1,2,2
3,1,2025-12-17,7,1,1,1
4,1,2025-12-18,7,2,1,1
...,...,...,...,...,...,...
879,10,2025-12-27,5,2,1,1
880,10,2025-12-28,2,3,2,1
881,10,2025-12-29,9,2,2,1
882,10,2025-12-30,5,2,2,2


In [138]:
# Creating content_usage DataFrame
def generate_content_usage(engagement_df):
    data =[]
    content_id = 1
    
    for i, row in engagement_df.iterrows():
        user_id = row['user_id']
        date = row['date']
        content_type = np.random.choice(["Audio", "Video", "Reading"])
        category = np.random.choice(["Anxiety", "Sleep", "Self-esteem", "Stress", "Relationships"])

        # simulating aleatory time spended for each category
        audio_minutes_consumed = np.random.randint(5,60)  #considering podcast are mainly 60 min long
        video_minutes_consumed = np.random.randint(5,90)  # considering video is one of the most engaging type of content
        reading_minutes_consumed = np.random.randint(5,45)  # Considering that reading is not too common inside an app

        if content_type == 'Audio':
            minutes_consumed = audio_minutes_consumed
        elif content_type == 'Video':
            minutes_consumed = video_minutes_consumed
        else :
            minutes_consumed = reading_minutes_consumed

        data.append({
                'user_id':user_id,
                'content_id': content_id,
                'date':date,
                'content_type':content_type,
                'category': category,
                'minutes_consumed': minutes_consumed
            })
        content_id += 1
    
    
    content_usage= pd.DataFrame(data) 
    return content_usage

In [109]:
# Creating outcomes DataFrame:
def generate_outcomes(users_df,sessions_df,assessments_df,engagement_df,content_usage_df):
    # Creating variables from other DataFrames
    sessions_variables = (
        sessions_df[sessions_df['attended'] == True].groupby('user_id').size().reset_index(name='total_sessions')
    )
    engagement_variables = (engagement_df.groupby('user_id').agg(
            total_active_days = ('date', 'nunique'),
            total_exercises_completed = ('exercises_completed', 'sum'),
            total_meditations_completed = ('meditations_completed', 'sum')))
    content_variables = (content_usage_df.groupby('user_id').agg(
            total_content_consumed_min = ('minutes_consumed', 'sum')))

    # Merging everything with users_df
    outcomes_df = users_df.merge(sessions_variables, on='user_id', how='left') \
                           .merge(engagement_variables, on='user_id', how='left') \
                           .merge(content_variables, on='user_id', how='left')
    # Fill NaNs (users with zero activity)
    outcomes_df[['total_sessions','total_active_days','total_exercises_completed','total_meditations_completed','total_content_consumed_min']] = (
    outcomes_df[['total_sessions','total_active_days','total_exercises_completed','total_meditations_completed','total_content_consumed_min']].fillna(0))

    # Calculating treatment intensity acording to the engagement, n sessions completed and active days
    treatment_intensity = (
        0.4 * outcomes_df['total_sessions'] +
        0.2 * outcomes_df['total_exercises_completed'] +
        0.2 * outcomes_df['total_meditations_completed'] +
        0.001 * outcomes_df['total_content_consumed_min'] +
        0.05 * outcomes_df['total_active_days']
    )

    # Simulating noise of external random real life situations
    noise = np.random.normal(0, 1, len(outcomes_df))
    
    # Generate final clinical outcomes
    outcomes_df['final_stress'] = (outcomes_df['baseline_stress'] - 0.05 * treatment_intensity + noise)
    outcomes_df['final_anxiety'] = (outcomes_df['baseline_anxiety'] - 0.06 * treatment_intensity + noise)
    outcomes_df['final_depression'] = (outcomes_df['baseline_depression'] - 0.04 * treatment_intensity + noise)
    outcomes_df['satisfaction_score'] = np.clip(3 + 0.15 * treatment_intensity - 0.3 * outcomes_df['dropout'].astype(int)
        + np.random.normal(0, 1, len(outcomes_df)), 2, 10)

    # creating protections in case of extreme outliers

    outcomes_df['final_stress'] = np.clip(outcomes_df['final_stress'], 2, 10)
    outcomes_df['final_anxiety'] = np.clip(outcomes_df['final_anxiety'], 2, 10)
    outcomes_df['final_depression'] = np.clip(outcomes_df['final_depression'], 2, 10)

    # reordering columns

    outcomes_df = outcomes_df[
        [
            'user_id',
            'dropout',
            'total_active_days',
            'total_sessions',
            'total_exercises_completed',
            'total_meditations_completed',
            'total_content_consumed_min',
            'baseline_stress',
            'final_stress',
            'baseline_anxiety',
            'final_anxiety',
            'baseline_depression',
            'final_depression',
            'satisfaction_score',
            'signup_date',
            'dropout_date'
        ]
    ]
    return outcomes_df
   

''' Please notice: calculations like final_stress or total_active_days, coulb be done later by the Analyst, 
    but I want to make things easier for the analysis and models from the beggining. If cleaning or reducing variables became too necesary
    you can always recode this final Dataset from the beggining too.
'''

' Please notice: calculations like final_stress or total_active_days, coulb be done later by the Analyst, \n    but I want to make things easier for the analysis and models from the beggining. If cleaning or reducing variables became too necesary\n    you can always recode this final Dataset from the beggining too.\n'

In [144]:

# Orchestration

def main():
    print("Generating AureaMind synthetic data...")
    print(f'{N_USERS} observations are being simulated')
    print(f'Start Date: {START_DATE} \nFinish Date: {FINISH_DATE}')

    users = generate_users()
    print(f'Users DataFrame conteins {len(users)} observations')
    assessments = generate_assessments(users)
    print(f'Assessments DataFrame conteins {len(assessments)} observations')
    sessions = generate_sessions(users)
    print(f'sessions DataFrame conteins {len(sessions)} observations')
    engagement = generate_engagement(users)
    print(f'engagementDataFrame conteins {len(engagement)} observations')
    content_usage = generate_content_usage(engagement)
    print(f'content_usage DataFrame conteins {len(content_usage)} observations')
    outcomes = generate_outcomes(users, sessions, assessments, engagement, content_usage)
    print(f'outcomes DataFrame conteins {len(outcomes)} observations')

    os.makedirs(OUTPUT_PATH, exist_ok=True)

    users.to_csv(f"{OUTPUT_PATH}/users.csv", index=False)
    assessments.to_csv(f"{OUTPUT_PATH}/assessments.csv", index=False)
    sessions.to_csv(f"{OUTPUT_PATH}/sessions.csv", index=False)
    engagement.to_csv(f"{OUTPUT_PATH}/engagement.csv", index=False)
    content_usage.to_csv(f"{OUTPUT_PATH}/content_usage.csv", index=False)
    outcomes.to_csv(f"{OUTPUT_PATH}/outcomes.csv", index=False)

    print("All datasets generated successfully.")  
    print(f'Datasets saved at {OUTPUT_PATH}')


if __name__ == "__main__":
    main()


Generating AureaMind synthetic data...
All datasets generated successfully.
